In [ ]:
import os
import shutil
import numpy as np

from theia.data_loading import load_trajectory_file
from theia.types import Point, Polarization, Radar
from theia.target_simulation.constant_radar_simulator import ConstantRadarSimulator
from theia.target_simulation.recorded_targets_simulator import RecordedTargetsSimulator
from theia.scenario_simulator import ScenarioSimulator
from pydantic import TypeAdapter

from theia.types import ActiveRadarDetection

# Configure paths

In [ ]:
trajectory_file = "../data/data_opensky_2022-06-27.csv"
output_dir = "../experiments/example_active"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Configure simulation

In [ ]:
# Load target trajectories (or ask user, generate dynamically, ...).
trajectories, callsigns = load_trajectory_file(trajectory_file)
for trajectory in trajectories:
    trajectory.cross_sections = np.ones(len(trajectory.times)) * 10.0
trajectories = [trajectory for trajectory in trajectories if len(trajectory.times) > 1]

# Define active radars.
radar = Radar(
    id=585,
    point=Point(
        lat=47.36700085728634,
        lon=8.537724304199216,
        alt=408,
    ),
    power=20000,
    erp=800,
    antenna_height=10.0,
    diameter=2.0,
    frequency=1000.0,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
    rotation_time=10.0,
    polarization=Polarization.HORIZONTAL,
)
radars = [radar]

# Define simulation.
target_simulator = RecordedTargetsSimulator(trajectories)
radar_simulator = ConstantRadarSimulator(
    radars=radars,
    on_time=target_simulator.get_minimum_time(),
    off_time=target_simulator.get_maximum_time(),
)
rng = np.random.Generator(np.random.PCG64(seed=45797093))
simulator = ScenarioSimulator(target_simulator, radar_simulator, rng)

# Run simulation

In [ ]:
detections = simulator.simulate_active_radar_detections()

# Save

In [ ]:
# Copy trajectories file.
shutil.copyfile(trajectory_file, f"{output_dir}/trajectories.csv")

# Save radars.
adapter_radars = TypeAdapter(list[Radar])
with open(f"{output_dir}/radars.json", "w") as file:
    file.write(adapter_radars.dump_json(radars).decode("utf-8"))

# Save detections.
adapter = TypeAdapter(list[ActiveRadarDetection])
with open(f"{output_dir}/detections.json", "w") as file:
    file.write(adapter.dump_json(detections).decode("utf-8"))